# Neural bigram model using embeddings

In this notebook we will modify our first version of the neural bigram model to use *embeddings* instead of one-hot vectors for representing characters.

## Dataset

We re-use the names dataset and our code for building the character index.

In [ ]:
# Load the list of names
with open("names-train.txt", encoding="utf-8") as f:
    names = [line.rstrip() for line in f]

# Build the character index
char2idx = {"$": 0}
for name in names:
    for char in name:
        if char not in char2idx:
            char2idx[char] = len(char2idx)

In [ ]:
def bigrams(names):
    for name in names:
        for x, y in zip("$" + name, name + "$"):
            yield x, y

We also use our code for storing all bigrams in a dataset:

In [ ]:
from torch.utils.data import Dataset


class ListDataset(list, Dataset):
    pass


train_bigrams = ListDataset(bigrams(names))

In [ ]:
len(train_bigrams)

## Embeddings

The major difference in our new model is the use of **embedding vectors** instead of one-hot vectors. Embeddings are covered in detail in Unit&nbsp;1 of the course. Here we only mention two important differences to one-hot vectors: First, embedding vectors contain floating-point numbers instead of integers. Second, the dimensionality of embedding vectors is chosen independently of the number of elements we want to encode. The one-hot vector had as many components as there are characters in our vocabulary – our embedding vectors will have a fixed size of&nbsp;16.

To use embeddings, we define our own model class:

In [ ]:
import torch.nn as nn


class Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        return self.fc(self.embedding(x))

This model encapsulates two layers: a linear layer (as before) and an **embedding layer**. The input to the `forward()` method is a batch of integer IDs representing previous characters. These IDs are sent through the embedding layer, which will replace each integer ID with a corresponding embedding vector. The dimensionality of that vector is determined in the `init()` method of the `Model` class. The output of the embedding layer is fed into the linear layer which produces logits (scores) for all possible next characters. This part of the network is therefore essentially identical to the linear model we used in the previous version of the notebook; the only difference is that it expects embedding vectors instead of one-hot vectors.

To use embedding vectors, we also change the `vectorize()` function:

In [ ]:
import torch
import torch.nn.functional as F


def vectorize(bigrams):
    # Split the batch into inputs (previous characters) and outputs (next characters)
    xs, ys = zip(*bigrams)

    # Replace each character by its integer index
    xs = [char2idx[x] for x in xs]
    ys = [char2idx[y] for y in ys]

    # Convert inputs and outputs into tensors
    tensor_x = torch.LongTensor(xs)  # do not make one-hot vectors
    tensor_y = torch.LongTensor(ys)

    return tensor_x, tensor_y

With this change, the vectorised representation of the first three bigrams are simply their indices in the character-to-index mapping:

In [ ]:
vectorize(train_bigrams[:3])

## Training the model

The rest of the code is essentially identical to the previous version, except that we use our custom model class instead of a linear layer.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
num_epochs = 10
batch_size = 64
lr = 1e-1

# Initialise the model
model = Model(len(char2idx), 16)

# Initialise the optimiser (stochastic gradient descent)
optimizer = optim.SGD(model.parameters(), lr=lr)

# Initialise the data loader
data_loader = DataLoader(
    train_bigrams,
    batch_size=batch_size,
    collate_fn=vectorize,
    shuffle=True,
)

# We train for the specified number of epochs
for _ in range(num_epochs):
    # In each epoch, we loop over the batches provided by the data loader
    for batch_x, batch_y in data_loader:
        # Reset the accumulated gradients
        optimizer.zero_grad()

        # Forward pass
        output = model.forward(batch_x)

        # Compute the loss
        loss = F.cross_entropy(output, batch_y)

        # Backward pass: propagate the loss and compute the gradients
        loss.backward()

        # Update the parameters
        optimizer.step()

Here is the embellished version of the training loop that plots the per-epoch losses.

In [ ]:
# Embellished training loop with plotting

import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import tqdm
from torch.utils.data import DataLoader

%config InlineBackend.figure_formats = ["svg"]

num_epochs = 10
batch_size = 64
lr = 1e-1

torch.manual_seed(42)

model = Model(len(char2idx), 16)
optimizer = optim.SGD(model.parameters(), lr=lr)
data_loader = DataLoader(
    train_bigrams,
    batch_size=batch_size,
    collate_fn=vectorize,
    shuffle=True,
)
losses = []
with tqdm.tqdm(total=num_epochs) as pbar:
    for _ in range(num_epochs):
        running_loss = 0
        for batch_x, batch_y in data_loader:
            optimizer.zero_grad()
            output = model.forward(batch_x)
            loss = F.cross_entropy(output, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        losses.append(running_loss / len(data_loader))
        pbar.update()
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Average loss")
plt.show()

## Evaluation

As before, we compute the perplexity of our model.

In [ ]:
with open("names-test.txt", encoding="utf-8") as f:
    test_names = [line.rstrip() for line in f]

test_bigrams = ListDataset(bigrams(test_names))

In [ ]:
# Do the following without gradient calculation
with torch.no_grad():
    # Get the vectorised version of all bigrams
    test_x, test_y = vectorize(test_bigrams)

    # Compute the cross-entropy loss (= average negative log likelihood)
    loss = F.cross_entropy(model.forward(test_x), test_y)

    # Convert to perplexity
    ppl = torch.exp(loss)

    # Print the perplexity
    print(f"{ppl:.1f}")

As we can see, the perplexity of the modified model is very similar to the value we observed with the model based on one-hot vectors, despite the fact that our embeddings have a fixed dimensionality.

That’s all folks!